# **Technologies for Anti-Money Laundering and Financial Crime**
# March 2025
### *COMP-1831-M01-2024-25*

This is a lab exercise focusing on blockchain transaction anonymity techniques and provides some simple visual examples using Neo4J.

## Python Dependencies

In [2]:
!pip install -U git+https://github.com/apogiatzis/etherscan_py
!pip install neo4j

  Cloning https://github.com/apogiatzis/etherscan_py to /tmp/pip-req-build-noywnhcm
  Running command git clone --filter=blob:none --quiet https://github.com/apogiatzis/etherscan_py /tmp/pip-req-build-noywnhcm
  Resolved https://github.com/apogiatzis/etherscan_py to commit 894eb45e20f3914a30457a92d4a92e53032b057d
  Preparing metadata (setup.py) ... done


## Neo4J Instance

For the purpose of visualizing some of the transactions in this lab, a Neo4J database will be used. Luckily Neo4J provides a free Neo4j instance that persists the data for 3 days before resetting. To follow along with the lab go ahead and create an account here(https://sandbox.neo4j.com/) to acquire your free instance.

**Use the credentials and urls provided in the code below to connect yo your instance.**

In [3]:
from neo4j import GraphDatabase

# change these to your credentials and url
BOLT_URL = "bolt://35.175.109.180:7687"
USER = "neo4j"
PWD = "rope-stuff-chattel"

class Neo4jConnection:

    def __init__(self, uri, user, pwd):
        self.__uri = uri
        self.__user = user
        self.__pwd = pwd
        self.__driver = None
        try:
            self.__driver = GraphDatabase.driver(self.__uri, auth=(self.__user, self.__pwd))
        except Exception as e:
            print("Failed to create the driver:", e)

    def close(self):
        if self.__driver is not None:
            self.__driver.close()

    def query(self, query, parameters=None, db=None):
        assert self.__driver is not None, "Driver not initialized!"
        session = None
        response = None
        try:
            session = self.__driver.session(database=db) if db is not None else self.__driver.session()
            response = list(session.run(query, parameters))
        except Exception as e:
            print("Query failed:", e)
        finally:
            if session is not None:
                session.close()
        return response


conn = Neo4jConnection(uri=BOLT_URL, user=USER, pwd=PWD)

In [4]:
conn.query('CREATE CONSTRAINT transactions IF NOT EXISTS FOR (t:Transaction) REQUIRE t.hash IS UNIQUE')
conn.query('CREATE CONSTRAINT addresses IF NOT EXISTS FOR (a:Address) REQUIRE a.public_key IS UNIQUE')

[]

### Cypher Queries:

Show all nodes and relationships:
```
MATCH (n) MATCH (n)-[r]-() RETURN n,r
```

Delete all nodes and relationships:
```
MATCH (n) MATCH (n)-[r]-() DELETE n,r
```

# Helper functions for Neo4J

In [5]:
def insert_data(query, rows, batch_size = 10000):
    # Function to handle the updating the Neo4j database in batch mode.

    total = 0
    batch = 0
    start = time.time()
    result = None

    while batch * batch_size < len(rows):

        res = conn.query(query,
                         parameters= {
                         'rows': rows[batch*batch_size:(batch+1)*batch_size].to_dict('records')})
        total += res[0]['total']
        batch += 1
        result = {"total":total,
                  "batches":batch,
                  "time":time.time()-start}
        print(result)

        return result

In [6]:
def add_transactions(address, transactions, direction="forward"):
    # Adds transactions nodes to the Neo4j graph.
    query = '''
      UNWIND $rows AS row
      MERGE (t:Transaction {txhash: row.txhash})
      ON CREATE SET
        t.gas_used = row.gas_used,
        t.value = row.value
      RETURN count(*) as total
    '''

    if direction == "forward":
      rel_query = """
        UNWIND $rows AS row
        MATCH
          (t:Transaction),
          (from_a:Address),
          (to_a:Address)
        WHERE t.txhash = row.txhash AND from_a.public_key = $address AND to_a.public_key = row.to_address
        CREATE (from_a)-[r_created:CREATED]->(t)
        CREATE (t)-[r_to:TO]->(to_a)
        RETURN  count(*) as total
      """
    elif direction == "backward":
      rel_query = """
        UNWIND $rows AS row
        MATCH
          (t:Transaction),
          (from_a:Address),
          (to_a:Address)
        WHERE t.txhash = row.txhash AND from_a.public_key = row.from_address AND to_a.public_key = $address
        CREATE (from_a)-[r_created:CREATED]->(t)
        CREATE (t)-[r_to:TO]->(to_a)
        RETURN  count(*) as total
      """

    transactions_added =  conn.query(query, parameters = {'rows':transactions})
    address_added =  conn.query(rel_query, parameters = {'rows':transactions, "address": address})

    return transactions_added

def add_addresses(addresses):
    # Adds addresses nodes to the Neo4j graph.
    query = '''
        UNWIND $rows AS row
        MERGE (a:Address {public_key: row})
        RETURN count(*) as total
    '''
    return conn.query(query, parameters = {'rows':addresses})

def save_to_neo4j(data, direction="forward"):
  # Create addresses
  add_addresses(list(data.keys()))

  # Create transactions
  for addr, transactions in data.items():
    add_transactions(addr, transactions, direction=direction)

# Etherscan Setup

For fetching Ethereum transactions we will use Etherscan's API. API has a rate limit of 5 requests per second so make sure you take this into consideration when requesting data.

You are encouraged to sign up at https://etherscan.io/ and create an API key.

In [7]:
from etherscan_py import etherscan_py

# Change this with your API key
ETHERSCAN_API_KEY="1XHSG9Z1HFIHP4QH798QH268SHAUDAXX3D"

sepolia_client = etherscan_py.Client(ETHERSCAN_API_KEY,network="sepolia")
mainnet_client = etherscan_py.Client(ETHERSCAN_API_KEY)

# Blockchain analysis functions

Below you can find some functions that follow transactions forward and backwards from a given address.

In [8]:
import time

addresses = {}

def forward_address_intel(client, address, depth=1, checked_addresses=set()):
  if depth == 0: return

  # Avoid rate limitting
  time.sleep(1.5)

  address = address.lower()
  print("Forward tracking from: ", address)

  # Get all transactions sent by that address
  transactions = client.get_all_transactions(from_address=address, status=2)

  outgoing_transactions = [t for t in transactions if t.from_address == address]

  addresses[address] = addresses.get(address, set())
  addresses[address] |= set(outgoing_transactions)
  checked_addresses.add(address)

  unchecked_addresses = set([t.to_address for t in outgoing_transactions]) - checked_addresses

  # Do the same thing on each distinct recipient address of the transactions
  for addr in unchecked_addresses:
    forward_address_intel(client=client, address=addr, depth=depth-1, checked_addresses=checked_addresses)

def backtrack_address_intel(client, address, depth=1, checked_addresses=set()):
  if depth == 0: return

  # Avoid rate limitting
  time.sleep(1.5)

  address = address.lower()
  print("Backtracking from: ", address)

  # Get all transactions sent by that address
  transactions = client.get_all_transactions(from_address=address, status=2)

  incoming_transactions = [t for t in transactions if t.to_address == address]
  # print(incoming_transactions)
  addresses[address] = addresses.get(address, set())
  addresses[address] |= set(incoming_transactions)
  checked_addresses.add(address)

  unchecked_addresses = set([t.from_address for t in incoming_transactions]) - checked_addresses

  # # Do the same thing on each distinct recipient address of the transactions
  for addr in unchecked_addresses:
    backtrack_address_intel(client=client, address=addr, depth=depth-1, checked_addresses=checked_addresses)

# Normal transaction backtracking

For the first scenario let's assume that we have been task to backtrack the origin of the funds (up to 3 levels deep) in address **0xC3d1AFCd0a9A3001Dbc2B671826C6322D0905bD7**. (Note that this is a Sepolia Testnet Ethereum address)

Verify that the source is: **0xF5A089235e4c099720c0E2E49a64d5ec14b28E0a**


In [9]:
addresses = {}
backtrack_address_intel(client=sepolia_client, address="0x8CC77838ca29F1299f68145FCe3a868f75432208", depth=3, checked_addresses=set())

Backtracking from:  0x8cc77838ca29f1299f68145fce3a868f75432208
Backtracking from:  0xda1f58738a2482423e589fb0a771bb6a6f9180fa
Backtracking from:  0x88da5c9147a4d3ff7a2f5cebdc8db2e83bd7c85b


In [10]:
# Convert transactions to dicts (Just for exporting to Neo4J)
backtrack_data_dict = {addr: [t.__dict__ for t in trans] for addr, trans in addresses.items()}

save_to_neo4j(backtrack_data_dict, direction="backward")

![](https://i.ibb.co/BB6SgkW/graph.png)

Now let's attempt the same analysis, with a mixer transaction executed using the https://tornado.cash/ service. Attempt to backtrack the funds in this Ethereum mainnet address: **0xc3219fAd2E51336424Cc2a83Bcd0AFF6372Be862**


In [ ]:
addresses = {}
backtrack_address_intel(client=mainnet_client, address="0xc3219fAd2E51336424Cc2a83Bcd0AFF6372Be862", depth=10, checked_addresses=set())

### What are Internal Transactions?

Simply put, internal transactions are transactions between contracts.

Internal transactions, despite the name, are not actual transactions. When two smart contracts "interact,'' the product or consequence of this interaction is a transaction. They are value transfers that were initiated by executing a smart contract (smart contract Ether or token transfer). They are a byproduct of smart contract functionality.

Source: https://support.bitpay.com/hc/en-us/articles/360050782511-What-are-Eth-Internal-Transactions-

In [ ]:
addresses = {}
forward_address_intel(client=mainnet_client, address="0xC85454C8eedf9811D1AEEFe98fDBAfd690E1937F", depth=10, checked_addresses=set())

In [ ]:
# Convert transactions to dicts (Just for exporting to Neo4J)
address={}
forward_data_dict = {addr: [t.__dict__ for t in trans] for addr, trans in addresses.items()}

save_to_neo4j(forward_data_dict, direction="forward")

![](https://i.ibb.co/dbqDwcH/graph-2.png)